In [ ]:
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import glob
import pickle

In [ ]:
γ_list     = ["0.079477","0.25133","0.79477","2.5133","7.9477","25.133","79.477","251.33"]
Q          = [316, 100, 31.6, 10, 3.16, 1, 0.316, 0.1]
all_states = [f"B{i}" for i in range(1,5)] + [f"D{i}" for i in range(1,11)]

In [ ]:
def latest_file(pattern):
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No file matching: {pattern}")
    print(files[-1])
    return files[-1]

In [ ]:
def read_tracedist(file):
    """Concatenate TraceDist across all 14 states for each gamma."""
    td_per_gamma = []
    for γ_i in γ_list:
        with h5py.File(file, "r") as f:
            td_all = []
            for state in all_states:
                td = f[γ_i][state]["TraceDist"][...]
                td_all.extend(td.tolist())
        td_per_gamma.append(td_all)
    return td_per_gamma

In [ ]:
kossak_td   = read_tracedist(latest_file("E_KOSSAK_TRACEDIST_ALLSTATES_*.h5"))
lindblad_td = read_tracedist(latest_file("E_LINDBLAD_TRACEDIST_ALLSTATES_*.h5"))

with open("NonMark.pkl", "rb") as fh:
    NonMark = pickle.load(fh)
NonMark

In [ ]:
color_blind_palette = ["#009E73", "#E69F00", "#CC79A7"]  # Green, Orange, Purple

def set_violin_colors(violin, color, alpha=0.5, zorder=2):
    for body in violin["bodies"]:
        body.set_facecolor(color); body.set_edgecolor("black")
        body.set_alpha(alpha);    body.set_zorder(zorder)
    for part in ["cbars", "cmins", "cmaxes"]:
        violin[part].set_zorder(zorder)
        violin[part].set_color(color)

plt.rcParams.update({"font.size": 14, "axes.labelsize": 16,
                     "xtick.labelsize": 14, "ytick.labelsize": 14,
                     "legend.fontsize": 14, "axes.titlesize": 18})

labels = []
fig, ax1 = plt.subplots(figsize=(8, 6))

# Kossakowski — strong coupling (low Q), positions 5-8
v1 = ax1.violinplot(kossak_td[4:], positions=[5, 6, 7, 8], widths=0.8)
set_violin_colors(v1, color_blind_palette[0], zorder=2)
labels.append((mpatches.Patch(color=color_blind_palette[0], alpha=0.5),
               "(a) - Kossakowski model"))

# Lindblad — weak coupling (high Q), positions 1-4
v2 = ax1.violinplot(lindblad_td[:4], positions=[1, 2, 3, 4], widths=0.8)
set_violin_colors(v2, color_blind_palette[1], zorder=2)
labels.append((mpatches.Patch(color=color_blind_palette[1], alpha=0.5),
               "(b) - Lindblad ansatz"))

# Non-Markovianity — twin right axis
ax2 = ax1.twinx()
ax2.plot(np.arange(1, 9), NonMark, marker="+", color="red",
         label=r"(c) - non-Markovianity $\mathcal{N}$")
ax2.set_yscale("log"); ax2.set_ylim(1e-6, .9)
ax2.set_ylabel(r"$\mathcal{N}$, non-Markovianity measure", color="red")
ax2.tick_params(axis="y", colors="red")
ax2.spines["right"].set_color("red")
ax2.legend(loc=1)

ax1.set_yscale("log"); ax1.set_ylim(1e-6, 1.0)
ax1.set_xticks(range(1, len(γ_list)+1), Q)
ax1.set_xlabel(r"$Q=\nu/\gamma$, quality factor")
ax1.set_ylabel(
    r"$T(\rho_{\mathrm{exact}},\rho_{\mathrm{SID}})$, trace distance")
ax1.legend(*zip(*labels), loc=2)

plt.tight_layout()
plt.show()
fig.savefig(
    "SB_SID_TRACEDIST_ALLSTATES_LINDBLAD&KOSSAK_vs_nonMarkovianity_color-blind.PDF")